# Open-Vocabulary Video Search First Progress Prototype

First progress prototype. Minimal end-to-end pipeline:

1. **Upload** an image **or** video.
2. **Extract** frames (a video is sampled; an image is a single frame).
3. **Segment** each frame with **SAM2** to get region crops.
4. **Encode** every crop + the whole frame with **SigLIP2**.
5. **Search** by text query against the in-memory embedding index.

Models are cached on Google Drive so they aren't re-downloaded between sessions.

Out of scope: vector DB, dedup, tracking, evaluation.

## 1. Install dependencies

In [ ]:
!pip install -q "open_clip_torch>=2.31.0" "timm>=1.0.15" opencv-python pillow numpy tqdm matplotlib ipywidgets
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

## 2. Mount Google Drive & route model caches there

On Colab this mounts `/content/drive` and points both the SAM2 checkpoint and the HuggingFace cache (used by SigLIP2) to a folder on your Drive. Off Colab it falls back to a local `data/cache` dir.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/spottr_proto")
except ImportError:
    DRIVE_ROOT = Path("data/cache")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = DRIVE_ROOT / "models"
HF_CACHE = DRIVE_ROOT / "hf"
MODEL_CACHE.mkdir(exist_ok=True)
HF_CACHE.mkdir(exist_ok=True)

# Must be set BEFORE huggingface_hub / open_clip pulls anything.
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE / "hub")

print(f"Drive root: {DRIVE_ROOT}")
print(f"SAM2 checkpoint dir: {MODEL_CACHE}")
print(f"HF cache: {HF_CACHE}")

## 3. Imports & config

In [ ]:
import urllib.request

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

FRAME_STRIDE_SECONDS = 1.0
MAX_FRAMES = 30
CROP_PAD_RATIO = 0.25
TOP_K = 5

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

## 4. Download SAM2 checkpoint (cached on Drive)

Using SAM 2.1 Hiera-Tiny (~38M params). Swap to `large` later for quality.

In [ ]:
SAM2_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"
SAM2_CHECKPOINT = MODEL_CACHE / "sam2.1_hiera_tiny.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_t.yaml"

if not SAM2_CHECKPOINT.exists():
    print(f"Downloading {SAM2_CHECKPOINT_URL}\n  -> {SAM2_CHECKPOINT}")
    urllib.request.urlretrieve(SAM2_CHECKPOINT_URL, str(SAM2_CHECKPOINT))
    print("Done.")
else:
    print(f"Cached: {SAM2_CHECKPOINT}")

## 5. Load SAM2

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_model = build_sam2(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=DEVICE)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=16,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=500,
)
print("SAM2 ready.")

## 6. Load SigLIP2

First run downloads weights to the Drive HF cache; subsequent runs load instantly.

In [ ]:
import open_clip

siglip_model, siglip_preprocess = open_clip.create_model_from_pretrained(
    "hf-hub:timm/ViT-B-16-SigLIP2-256"
)
siglip_model = siglip_model.to(DEVICE).eval()
siglip_tokenizer = open_clip.get_tokenizer("hf-hub:timm/ViT-B-16-SigLIP2-256")

with torch.no_grad():
    probe = siglip_tokenizer(["probe"]).to(DEVICE)
    SIGLIP_DIM = siglip_model.encode_text(probe).shape[-1]
print(f"SigLIP2 ready. dim={SIGLIP_DIM}")


def encode_images_siglip(pil_images, batch_size=32):
    if not pil_images:
        return np.zeros((0, SIGLIP_DIM), dtype=np.float32)
    out = []
    for i in range(0, len(pil_images), batch_size):
        batch = torch.stack([siglip_preprocess(im) for im in pil_images[i:i+batch_size]]).to(DEVICE)
        with torch.no_grad():
            emb = siglip_model.encode_image(batch)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32)


def encode_text_siglip(texts):
    tokens = siglip_tokenizer(texts).to(DEVICE)
    with torch.no_grad():
        emb = siglip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy().astype(np.float32)

## 7. Point to your input file

Upload your image or video into Colab's left-side **Files** panel (or `!cp` it from Drive), then set `INPUT_PATH` below to its location. Works with any image or video extension listed in `IMAGE_EXTS` / `VIDEO_EXTS`.

In [ ]:
INPUT_PATH = Path("data/input.mp4")  # <-- edit this to your file

ext = INPUT_PATH.suffix.lower()
assert INPUT_PATH.exists(), f"File not found: {INPUT_PATH.resolve()}"
assert ext in IMAGE_EXTS | VIDEO_EXTS, f"Unsupported extension: {ext}"
print(f"Input: {INPUT_PATH}  ({'image' if ext in IMAGE_EXTS else 'video'}, "
      f"{INPUT_PATH.stat().st_size / 1e6:.1f} MB)")

## 8. Load frames (image **or** video)

An image becomes a single-frame list; a video is sampled every `FRAME_STRIDE_SECONDS`.

In [ ]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    return [img], [0.0], None


def load_video(path, stride_seconds=1.0, max_frames=30):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, int(round(fps * stride_seconds)))

    frames, timestamps = [], []
    for frame_idx in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, bgr = cap.read()
        if not ok:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(rgb))
        timestamps.append(frame_idx / fps)
        if len(frames) >= max_frames:
            break
    cap.release()
    return frames, timestamps, fps


def load_frames(path):
    ext = Path(path).suffix.lower()
    if ext in IMAGE_EXTS:
        return load_image(path)
    if ext in VIDEO_EXTS:
        return load_video(path, FRAME_STRIDE_SECONDS, MAX_FRAMES)
    raise ValueError(f"Unsupported file type: {ext}")


frames, timestamps, fps = load_frames(INPUT_PATH)
kind = "image" if fps is None else f"video @ {fps:.1f} fps"
print(f"Loaded {len(frames)} frame(s) from {INPUT_PATH.name} ({kind})")

## 9. Build the in-memory index

Per frame: SAM2 masks → padded crops + whole frame → SigLIP2 embeddings stacked into one `(N, D)` numpy array. `region_to_frame` maps each row back to its source frame.

In [ ]:
def crops_and_boxes_from_masks(pil_image, masks, pad_ratio=CROP_PAD_RATIO, min_side=10):
    arr = np.array(pil_image)
    H, W = arr.shape[:2]
    crops, boxes = [], []
    for m in masks:
        x, y, w, h = [int(v) for v in m["bbox"]]
        pad_w, pad_h = int(w * pad_ratio), int(h * pad_ratio)
        x1, y1 = max(0, x - pad_w), max(0, y - pad_h)
        x2, y2 = min(W, x + w + pad_w), min(H, y + h + pad_h)
        if x2 - x1 < min_side or y2 - y1 < min_side:
            continue
        crops.append(Image.fromarray(arr[y1:y2, x1:x2]))
        boxes.append((x1, y1, x2 - x1, y2 - y1))
    crops.append(pil_image)              # whole frame as a region
    boxes.append((0, 0, W, H))
    return crops, boxes


all_embeddings = []
region_to_frame = []
region_boxes = []

for f_idx, frame in enumerate(tqdm(frames, desc="Indexing frames")):
    masks = mask_generator.generate(np.array(frame))
    crops, boxes = crops_and_boxes_from_masks(frame, masks)
    embs = encode_images_siglip(crops)
    all_embeddings.append(embs)
    region_to_frame.extend([f_idx] * embs.shape[0])
    region_boxes.extend(boxes)

index = np.vstack(all_embeddings).astype(np.float32) if all_embeddings else np.zeros((0, SIGLIP_DIM), dtype=np.float32)
region_to_frame = np.array(region_to_frame, dtype=np.int64)
region_boxes = np.array(region_boxes, dtype=np.int64)  # (N, 4) xywh

print(f"Index: {index.shape[0]} regions across {len(frames)} frame(s) "
      f"(avg {index.shape[0]/max(1,len(frames)):.1f} regions/frame)")

## 10. Search

In [ ]:
import matplotlib.patches as patches


def search(query, top_k=TOP_K):
    text_emb = encode_text_siglip([query])[0]
    sims = index @ text_emb
    best_score = np.full(len(frames), -np.inf, dtype=np.float32)
    best_region = np.full(len(frames), -1, dtype=np.int64)
    for region_idx, frame_idx in enumerate(region_to_frame):
        if sims[region_idx] > best_score[frame_idx]:
            best_score[frame_idx] = sims[region_idx]
            best_region[frame_idx] = region_idx
    order = np.argsort(-best_score)[:top_k]
    return [
        (int(i), float(best_score[i]), float(timestamps[i]), int(best_region[i]))
        for i in order
    ]


def show_results(query, top_k=TOP_K):
    hits = search(query, top_k=top_k)
    n = len(hits)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (frame_idx, score, ts, region_idx) in zip(axes, hits):
        frame = frames[frame_idx]
        ax.imshow(frame)
        x, y, w, h = region_boxes[region_idx]
        is_whole = (w >= frame.size[0] and h >= frame.size[1])
        edge = "yellow" if is_whole else "lime"
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor=edge, linewidth=2.5))
        title = f"frame {frame_idx}"
        if fps is not None:
            title += f" @ {ts:.1f}s"
        tag = " (whole frame)" if is_whole else " (region)"
        ax.set_title(f"{title}\nscore={score:.3f}{tag}")
        ax.axis("off")
    fig.suptitle(f'query: "{query}"', fontsize=14)
    plt.tight_layout()
    plt.show()

## 11. Run a query

Edit the `query` string and re-run the cell.

In [ ]:
query = "a person walking"
show_results(query, top_k=5)